# Spark Structured Streaming — 实时区域需求聚合 + 离线对账

Kafka `tlc-trips` → 解析 JSON → **与 DWD 完全相同的清洗规则** → 5 分钟事件时间滚动窗口 × 上车区域聚合 → `foreachBatch` 写 Parquet，并与离线 DWD 同窗口逐区域对账。

| 配置 | 作用 |
|---|---|
| `withWatermark("pickup_datetime", "2 minutes")` | 最多等待迟到 2 分钟的数据，之后窗口关闭 |
| `outputMode("append")` | 每个窗口关闭后只输出一次，不会重复写入 |
| `checkpointLocation` | 记录 Kafka offset 与聚合状态，重启可续跑 |
| heartbeat（虚拟区域 `-1`） | 推进水位线、关闭最后的窗口；不能在聚合前 filter（优化器会把过滤下推到水位线之前），在 `foreachBatch` 里排除 |

运行顺序：`bash scripts/reset_stream.sh` → 启动本 notebook → 另一个终端运行 08。本 notebook 收到数据后，持续 30 秒没有新数据即自动停止。

In [ ]:
import os, shutil, time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType

spark = (
    SparkSession.builder.appName("TLC-Streaming")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "4g")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE         = os.path.expanduser("~/tlc-spark-pipeline")
DWD_FACT     = f"{BASE}/data/warehouse/dwd/fact_trips"
STREAM_DIR   = f"{BASE}/data/streaming"
STREAM_OUT   = f"{STREAM_DIR}/zone_5min"
RECON_OUT    = f"{STREAM_DIR}/recon_batches"
CHECKPOINT   = f"{STREAM_DIR}/checkpoint"
REPLAY_START = os.environ.get("REPLAY_START", "2024-03-15 08:00:00")
REPLAY_END   = os.environ.get("REPLAY_END",   "2024-03-15 09:00:00")
MAX_WAIT_SEC = int(os.environ.get("STREAM_MAX_WAIT", "900"))
HEARTBEAT_ZONE = -1

# 每次运行从干净状态开始：输出用 append 写，残留会导致重复计数
shutil.rmtree(STREAM_DIR, ignore_errors=True)
print(f"Spark {spark.version} | replay window [{REPLAY_START}, {REPLAY_END})")

## 1. 离线基准：DWD fact_trips 同时段、同口径的 5 分钟 × 区域计数

In [ ]:
d = REPLAY_START[:10]
y, m, dd = int(d[:4]), int(d[5:7]), int(d[8:10])
offline = (
    spark.read.parquet(DWD_FACT)
    .filter((F.col("source_type") == "yellow") & (F.col("pickup_year") == y)
            & (F.col("pickup_month") == m) & (F.col("pickup_day") == dd))
    .filter((F.col("pickup_datetime") >= F.lit(REPLAY_START).cast("timestamp")) &
            (F.col("pickup_datetime") <  F.lit(REPLAY_END).cast("timestamp")))
    .groupBy(F.window("pickup_datetime", "5 minutes").start.alias("window_start"), "PULocationID")
    .agg(F.count("*").alias("offline_count"))
    .cache()
)
print(f"Offline baseline: {offline.count():,} (window × zone) cells, "
      f"{offline.agg(F.sum('offline_count')).first()[0]:,} trips")

## 2. 读取 Kafka 流并解析

In [ ]:
schema = StructType([
    StructField("VendorID",         IntegerType()),
    StructField("pickup_datetime",  StringType()),
    StructField("dropoff_datetime", StringType()),
    StructField("PULocationID",     IntegerType()),
    StructField("DOLocationID",     IntegerType()),
    StructField("passenger_count",  IntegerType()),
    StructField("trip_distance",    DoubleType()),
    StructField("fare_amount",      DoubleType()),
    StructField("tip_amount",       DoubleType()),
    StructField("total_amount",     DoubleType()),
    StructField("payment_type",     IntegerType()),
    StructField("source_type",      StringType()),
    StructField("is_heartbeat",     BooleanType()),
])

TS = "yyyy-MM-dd HH:mm:ss"
parsed = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "tlc-trips")
    .option("startingOffsets", "earliest")
    .load()
    .select(F.from_json(F.col("value").cast("string"), schema).alias("d"))
    .select("d.*")
    .withColumn("pickup_datetime",  F.to_timestamp("pickup_datetime",  TS))
    .withColumn("dropoff_datetime", F.to_timestamp("dropoff_datetime", TS))
)

## 3. 清洗（与 03_dwd_build 完全一致）+ 窗口聚合

清洗写法刻意与 DWD 相同：先用 `when` 链标注 `dq_reason`，再保留 `dq_reason IS NULL` 的记录。
如果改成 `filter(fare > 0)` 这类写法，遇到 null 时结果会和离线不同（SQL 三值逻辑），对账就会出现差异。

In [ ]:
col, ut = F.col, F.unix_timestamp
secs = ut("dropoff_datetime") - ut("pickup_datetime")

cleaned = (
    parsed
    .withWatermark("pickup_datetime", "2 minutes")
    # 注意：不能在这里 filter 掉 heartbeat —— 优化器会把过滤下推到水位线之前，
    # heartbeat 就无法推进水位线。heartbeat 作为合法记录聚合进区域 -1，在 foreachBatch 中排除。
    .withColumn("trip_duration_min", F.round(secs / 60, 2))
    .withColumn("avg_speed_mph",
        F.when(ut("dropoff_datetime") > ut("pickup_datetime"),
               F.round(col("trip_distance") / (secs / 3600), 2)).otherwise(F.lit(None)))
    .withColumn("dq_reason",
        F.when(col("fare_amount") <= 0, "non_positive_fare")
        .when(col("trip_distance") <= 0, "non_positive_trip_distance")
        .when(col("dropoff_datetime") <= col("pickup_datetime"), "invalid_time_order")
        .when(col("PULocationID").isNull(), "null_PULocationID")
        .when(col("DOLocationID").isNull(), "null_DOLocationID")
        .when(col("passenger_count").isNotNull() & ~col("passenger_count").between(1, 6), "invalid_passenger_count")
        .when((col("pickup_datetime") < "2024-01-01") | (col("pickup_datetime") >= "2024-04-01"), "out_of_range_datetime")
        .when(col("avg_speed_mph").isNotNull() & (col("avg_speed_mph") > 100), "excessive_speed")
        .when(~col("trip_duration_min").between(1, 180), "invalid_duration"))
    .filter(col("dq_reason").isNull())
)

zone_5min = (
    cleaned
    .groupBy(F.window("pickup_datetime", "5 minutes"), "PULocationID")
    .agg(F.count("*").alias("trip_count"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
         F.round(F.avg("trip_duration_min"), 1).alias("avg_duration_min"))
    .select(F.col("window.start").alias("window_start"),
            F.col("window.end").alias("window_end"),
            "PULocationID", "trip_count", "avg_fare", "avg_duration_min")
)

## 4. foreachBatch：写出窗口结果 + 每批与离线对账

In [ ]:
def write_batch(batch_df, batch_id):
    batch_df = batch_df.filter(F.col("PULocationID") != HEARTBEAT_ZONE)   # 排除 heartbeat
    if batch_df.isEmpty():
        return
    batch_df.persist()
    batch_df.withColumn("batch_id", F.lit(batch_id)).write.mode("append").parquet(STREAM_OUT)

    recon = (batch_df.select("window_start", "PULocationID", "trip_count")
             .join(offline, ["window_start", "PULocationID"], "left")
             .fillna(0, ["offline_count"])
             .withColumn("diff", F.col("trip_count") - F.col("offline_count"))
             .withColumn("batch_id", F.lit(batch_id)))
    recon.write.mode("append").parquet(RECON_OUT)

    s = recon.agg(F.count("*").alias("cells"), F.sum("trip_count").alias("stream"),
                  F.sum("offline_count").alias("offline"),
                  F.sum((F.col("diff") != 0).cast("int")).alias("mismatch")).first()
    print(f"batch {batch_id}: closed cells={s.cells}  stream={s.stream}  offline={s.offline}  mismatched cells={s.mismatch}")
    batch_df.unpersist()

query = (
    zone_5min.writeStream
    .outputMode("append")
    .foreachBatch(write_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(processingTime="10 seconds")
    .start()
)
print("Streaming query started — waiting for data (run 08_kafka_producer now)...")

## 5. 等待：收到数据后持续 30 秒无新数据即停止

In [ ]:
start, idle = time.time(), 0
while query.isActive and time.time() - start < MAX_WAIT_SEC:
    time.sleep(5)
    got_data = sum(p["numInputRows"] for p in query.recentProgress) > 0
    st = query.status
    idle = idle + 1 if (got_data and not st["isDataAvailable"] and not st["isTriggerActive"]) else 0
    if idle >= 6:
        break
query.stop()
print(f"Stopped after {time.time()-start:.0f}s, batches={len(query.recentProgress)}, "
      f"input rows={sum(p['numInputRows'] for p in query.recentProgress):,}")

## 6. 最终对账：实时 vs 离线（全外连接，逐 5 分钟 × 区域）

全外连接能同时发现两类问题：实时多算的格子，以及离线有、实时漏掉的格子。

In [ ]:
stream = (spark.read.parquet(STREAM_OUT).filter(F.col("PULocationID") != HEARTBEAT_ZONE)
          .select("window_start", "PULocationID", "trip_count"))
dup = stream.groupBy("window_start", "PULocationID").count().filter("count > 1").count()

cmp = (stream.join(offline, ["window_start", "PULocationID"], "full_outer")
       .fillna(0, ["trip_count", "offline_count"])
       .withColumn("diff", F.col("trip_count") - F.col("offline_count")))

t = cmp.agg(F.count("*").alias("cells"),
            F.sum("trip_count").alias("stream"),
            F.sum("offline_count").alias("offline"),
            F.sum((F.col("diff") != 0).cast("int")).alias("mismatch"),
            F.countDistinct("window_start").alias("windows")).first()
diff_pct = abs(t.stream - t.offline) / t.offline * 100 if t.offline else 0

print(f"Replay window       : [{REPLAY_START}, {REPLAY_END})  ({t.windows} × 5-min windows)")
print(f"(window × zone) cells: {t.cells:,}")
print(f"Streaming trips     : {t.stream:,}")
print(f"Offline DWD trips   : {t.offline:,}")
print(f"Total diff          : {abs(t.stream - t.offline):,}  ({diff_pct:.3f}%)")
print(f"Mismatched cells    : {t.mismatch}")
print(f"Duplicate cells     : {dup}  (应为 0：append 模式每个窗口只输出一次)")

if t.mismatch:
    cmp.filter("diff != 0").orderBy("window_start", "PULocationID").show(20, False)
print("\n✅ 实时与离线完全一致" if t.mismatch == 0 and dup == 0 else "\n⚠️ 存在差异，见上表")

In [ ]:
spark.stop()
print("Spark stopped.")